# Contextual based neural lemmatization with Flair

In [53]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [54]:
!pip install syntok

In [55]:
!pip install flair

In [56]:
from flair.data import Corpus
from flair.datasets import ColumnCorpus
from flair.embeddings import TokenEmbeddings, WordEmbeddings, StackedEmbeddings, FlairEmbeddings, CharacterEmbeddings
from typing import List

# ASSIGNMENT 1

Please inspect the data we provided (train, dev and test sets available): 'content/drive/My Drive/Colab Notebooks/2025-ILTAPP/datasets/sigmorphon2019/'

+ TODO Try to spot the differences with the Lab 04-ner-training-tagging_TODO.ipynb.
+ TODO: define the columns required to instantiate the ColumnCorpus and the tag we want to predict.
+ TODO: use downsampling to speed up training.
+ TODO: print the total number of tags to predict in the corpus and check how this changes as you change the downsampling.
+ TODO: experiment with Classic Embeddings, CharacterEmbeddings and Flair Embeddings for improving accuracy performance (also with number of epochs).
  + HINT: Do not combine more than two types of embeddings to avoid running out RAM.

In [57]:
# TODO define columns
# The sigmorphon data has: token, lemma_rule (and possibly POS/other cols)
# Looking at the expected output, column 0 = token, column 1 = lemma_rule (the tag)
# Change the index from 1 to 2
columns = {0: 'text', 2: 'lemma'}
#

# TODO get the corpus
# Use downsampling (0.1 = 10%) to speed up training
corpus: Corpus = ColumnCorpus(
    data_folder='', # Data folder is empty as full paths are provided
    column_format=columns, # Pass columns as column_format positional argument
    train_file='/content/drive/MyDrive/Colab Notebooks/2026-ILTAPP/en-ewt-ud-ses-true-train.tsv',
    dev_file='/content/drive/MyDrive/Colab Notebooks/2026-ILTAPP/en-ewt-ud-ses-true-dev.tsv',
    test_file='/content/drive/MyDrive/Colab Notebooks/2026-ILTAPP/en-ewt-ud-ses-true-test.tsv',
    # column_delimiter='\t'  # TSV files are tab-separated by default
).downsample(0.1)

print(corpus)

2026-03-04 15:53:39,741 Reading data from .
2026-03-04 15:53:39,742 Train: /content/drive/MyDrive/Colab Notebooks/2026-ILTAPP/en-ewt-ud-ses-true-train.tsv
2026-03-04 15:53:39,744 Dev: /content/drive/MyDrive/Colab Notebooks/2026-ILTAPP/en-ewt-ud-ses-true-dev.tsv
2026-03-04 15:53:39,745 Test: /content/drive/MyDrive/Colab Notebooks/2026-ILTAPP/en-ewt-ud-ses-true-test.tsv
Corpus: 1330 train + 166 dev + 166 test sentences


In [58]:
# TODO what tag do we want to predict?
label_type = 'lemma'

# 3. make the tag dictionary from the corpus
tag_dictionary = corpus.make_label_dictionary(label_type=label_type, add_unk=True)
print(tag_dictionary.idx2item)

# TODO print total number of tags
print(f"Total number of tags: {len(tag_dictionary)}")

2026-03-04 15:53:45,062 Computing label dictionary. Progress:


0it [00:00, ?it/s]
1330it [00:00, 21102.34it/s]

2026-03-04 15:53:45,132 Dictionary created for label 'lemma' with 102 values: ↓0;d¦+ (seen 16150 times), ↑0¦↓1;d¦+ (seen 1493 times), ↓0;d¦-+ (seen 1173 times), ↓0;abe (seen 428 times), ↓0;d¦--+ (seen 300 times), ↓0;d¦---+ (seen 219 times), ↓0;d--+b¦+ (seen 130 times), ↑0¦↓-1;d¦+ (seen 115 times), ↓0;d¦-+v+e+ (seen 112 times), ↓0;d¦---+e+ (seen 100 times), ↓0;d¦-+o→+ (seen 75 times), ↓0;d¦--+e+ (seen 74 times), ↓0;d¦--+y+ (seen 69 times), ↑0¦↓1;ai (seen 68 times), ↓0;d¦---+y+ (seen 63 times), ↓0;awe (seen 42 times), ↓0;d¦-+y+ (seen 38 times), ↓0;d-+b¦--+ (seen 38 times), ↓0;d¦----+ (seen 27 times), ↓0;d¦--+o+ (seen 26 times)
[b'<unk>', b'\xe2\x86\x930;d\xc2\xa6+', b'\xe2\x86\x910\xc2\xa6\xe2\x86\x931;d\xc2\xa6+', b'\xe2\x86\x930;d\xc2\xa6-+', b'\xe2\x86\x930;abe', b'\xe2\x86\x930;d\xc2\xa6--+', b'\xe2\x86\x930;d\xc2\xa6---+', b'\xe2\x86\x930;d--+b\xc2\xa6+', b'\xe2\x86\x910\xc2\xa6\xe2\x86\x93-1;d\xc2\xa6+', b'\xe2\x86\x930;d\xc2\xa6-+v+e+', b'\xe2\x86\x930;d\xc2\xa6---+e+', b'\xe2\x86

In [59]:
pip install flair[word-embeddings]

In [60]:
# TODO: experiment with different embeddings and check performance; find the best combination
from flair.embeddings import WordEmbeddings, FlairEmbeddings, StackedEmbeddings
from typing import List

embedding_types: List = [
    # Character-level embeddings
    # CharacterEmbeddings(),

    # Pre-trained word embeddings (English)
    WordEmbeddings('glove'),

    # Contextual Flair embeddings (forward and backward)
    FlairEmbeddings('news-forward', chars_per_chunk=128),
    FlairEmbeddings('news-backward', chars_per_chunk=128),
]

# Stack all embeddings into a single object
embeddings: StackedEmbeddings = StackedEmbeddings(embeddings=embedding_types)


In [61]:
# initialize sequence tagger
from flair.models import SequenceTagger

tagger: SequenceTagger = SequenceTagger(hidden_size=256,
                                        embeddings=embeddings,
                                        tag_dictionary=tag_dictionary,
                                        tag_type=label_type
)

2026-03-04 15:53:55,306 SequenceTagger predicts: Dictionary with 102 tags: <unk>, ↓0;d¦+, ↑0¦↓1;d¦+, ↓0;d¦-+, ↓0;abe, ↓0;d¦--+, ↓0;d¦---+, ↓0;d--+b¦+, ↑0¦↓-1;d¦+, ↓0;d¦-+v+e+, ↓0;d¦---+e+, ↓0;d¦-+o→+, ↓0;d¦--+e+, ↓0;d¦--+y+, ↑0¦↓1;ai, ↓0;d¦---+y+, ↓0;awe, ↓0;d¦-+y+, ↓0;d-+b¦--+, ↓0;d¦----+, ↓0;d¦--+o+, ↓0;d+s¦-+, ↓0;d-+w+i¦+, ↓0;d-+h+a¦+, ↓0;d¦-+e→+, ↓0;d¦--+a→+e+, ↓0;d¦-+k→+, ↓0;d→--+i¦+, ↓0;d¦--+a+v+e+, ↓0;ago, ↓0;d→-+i¦+, ↓0;d→-+o¦+, ↓0;d¦+n+, ↓0;d¦-----+i+n+k+, ↓0;d¦+e→-+, ↓0;d+'¦+, ↓0;d¦-+i+l+l+, ↓0;d¦-+d+, ↓0;d¦-+i→+, ↓0;d¦-+e→-+l+, ↑0¦↓1¦↑2¦↓3;d¦+, ↓0;d¦--+i+n→--+, ↑0¦↓1¦↑2¦↓-2;d¦+, ↓0;a", ↓0;d¦-+a→+, ↓0;d¦+o→+, ↓0;d¦-+u→+, ↓0;d-+'¦+, ↓0;a-, ↓0;d¦+a→+


In [62]:
# initialize trainer
from flair.trainers import ModelTrainer
trainer: ModelTrainer = ModelTrainer(tagger, corpus)

In [63]:
trainer.train('/content/drive/My Drive/Colab Notebooks/2026-ILTAPP/en-lemma-model', mini_batch_size=16, train_with_dev=False, max_epochs=5)

2026-03-04 15:53:55,570 ----------------------------------------------------------------------------------------------------
2026-03-04 15:53:55,572 Model: "SequenceTagger(
  (embeddings): StackedEmbeddings(
    (list_embedding_0): WordEmbeddings(
      'glove'
      (embedding): Embedding(400001, 100)
    )
    (list_embedding_1): FlairEmbeddings(
      (lm): LanguageModel(
        (drop): Dropout(p=0.05, inplace=False)
        (encoder): Embedding(300, 100)
        (rnn): LSTM(100, 2048)
      )
    )
    (list_embedding_2): FlairEmbeddings(
      (lm): LanguageModel(
        (drop): Dropout(p=0.05, inplace=False)
        (encoder): Embedding(300, 100)
        (rnn): LSTM(100, 2048)
      )
    )
  )
  (word_dropout): WordDropout(p=0.05)
  (locked_dropout): LockedDropout(p=0.5)
  (embedding2nn): Linear(in_features=4196, out_features=4196, bias=True)
  (rnn): LSTM(4196, 256, batch_first=True, bidirectional=True)
  (linear): Linear(in_features=512, out_features=104, bias=True)
  (loss_

/usr/local/lib/python3.12/dist-packages/flair/trainers/trainer.py:107: UserWarning: There should be no best model saved at epoch 1 except there is a model from previous trainings in your training folder. All previous best models will be deleted.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/flair/trainers/trainer.py:545: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp and flair.device.type != "cpu")


2026-03-04 15:53:58,893 epoch 1 - iter 8/84 - loss 2.80075465 - time (sec): 3.29 - samples/sec: 654.89 - lr: 0.100000 - momentum: 0.000000
2026-03-04 15:54:01,940 epoch 1 - iter 16/84 - loss 2.16746440 - time (sec): 6.33 - samples/sec: 671.80 - lr: 0.100000 - momentum: 0.000000
2026-03-04 15:54:04,657 epoch 1 - iter 24/84 - loss 1.96453937 - time (sec): 9.05 - samples/sec: 691.55 - lr: 0.100000 - momentum: 0.000000
2026-03-04 15:54:07,337 epoch 1 - iter 32/84 - loss 1.83409639 - time (sec): 11.73 - samples/sec: 693.66 - lr: 0.100000 - momentum: 0.000000
2026-03-04 15:54:10,248 epoch 1 - iter 40/84 - loss 1.73044946 - time (sec): 14.64 - samples/sec: 706.53 - lr: 0.100000 - momentum: 0.000000
2026-03-04 15:54:12,657 epoch 1 - iter 48/84 - loss 1.65379858 - time (sec): 17.05 - samples/sec: 717.37 - lr: 0.100000 - momentum: 0.000000
2026-03-04 15:54:15,564 epoch 1 - iter 56/84 - loss 1.59239538 - time (sec): 19.96 - samples/sec: 711.86 - lr: 0.100000 - momentum: 0.000000
2026-03-04 15:54:

100%|██████████| 3/3 [00:02<00:00,  1.19it/s]

2026-03-04 15:54:27,271 DEV : loss 0.747768759727478 - f1-score (micro avg)  0.8278
2026-03-04 15:54:27,281  - 0 epochs without improvement
2026-03-04 15:54:27,294 saving best model


...

2026-03-04 15:55:15,139 DEV : loss 0.23314660787582397 - f1-score (micro avg)  0.9382
2026-03-04 15:55:15,149  - 0 epochs without improvement
2026-03-04 15:55:15,176 saving best model


2026-03-04 15:55:18,487 ----------------------------------------------------------------------------------------------------
2026-03-04 15:55:18,526 Loading model from best epoch ...
2026-03-04 15:55:20,832 SequenceTagger predicts: Dictionary with 104 tags: <unk>, ↓0;d¦+, ↑0¦↓1;d¦+, ↓0;d¦-+, ↓0;abe, ↓0;d¦--+, ↓0;d¦---+, ↓0;d--+b¦+, ↑0¦↓-1;d¦+, ↓0;d¦-+v+e+, ↓0;d¦---+e+, ↓0;d¦-+o→+, ↓0;d¦--+e+, ↓0;d¦--+y+, ↑0¦↓1;ai, ↓0;d¦---+y+, ↓0;awe, ↓0;d¦-+y+, ↓0;d-+b¦--+, ↓0;d¦----+, ↓0;d¦--+o+, ↓0;d+s¦-+, ↓0;d-+w+i¦+, ↓0;d-+h+a¦+, ↓0;d¦-+e→+, ↓0;d¦--+a→+e+, ↓0;d¦-+k→+, ↓0;d→--+i¦+, ↓0;d¦--+a+v+e+, ↓0;ago, ↓0;d→-+i¦+, ↓0;d→-+o¦+, ↓0;d¦+n+, ↓0;d¦-----+i+n+k+, ↓0;d¦+e→-+, ↓0;d+'¦+, ↓0;d¦-+i+l+l+, ↓0;d¦-+d+, ↓0;d¦-+i→+, ↓0;d¦-+e→-+l+, ↑0¦↓1¦↑2¦↓3;d¦+, ↓0;d¦--+i+n→--+, ↑0¦↓1¦↑2¦↓-2;d¦+, ↓0;a", ↓0;d¦-+a→+, ↓0;d¦+o→+, ↓0;d¦-+u→+, ↓0;d-+'¦+, ↓0;a-, ↓0;d¦+a→+


100%|██████████| 3/3 [00:01<00:00,  1.66it/s]

2026-03-04 15:55:23,047 
Results:
- F-score (micro) 0.934
- F-score (macro) 0.3017
- Accuracy 0.934

By class:
                 precision    recall  f1-score   support

         ↓0;d¦+     0.9656    0.9884    0.9769      2074
      ↑0¦↓1;d¦+     0.8730    0.9066    0.8895       182
        ↓0;d¦-+     0.7817    0.8538    0.8162       130
         ↓0;abe     0.9800    0.9423    0.9608        52
       ↓0;d¦--+     0.5366    0.5366    0.5366        41
      ↓0;d¦---+     0.6579    0.7812    0.7143        32
     ↓0;d--+b¦+     0.7500    1.0000    0.8571        21
     ↑0¦↓-1;d¦+     0.7647    0.5417    0.6341        24
    ↓0;d¦-+v+e+     0.8000    1.0000    0.8889        12
     ↓0;d¦-+o→+     1.0000    0.9000    0.9474        10
     ↓0;d¦--+y+     1.0000    1.0000    1.0000         9
         ↓0;awe     1.0000    0.0714    0.1333        14
     ↓0;d¦--+e+     0.8750    1.0000    0.9333         7
    ↓0;d-+b¦--+     1.0000    0.3000    0.4615        10
       ↑0¦↓1;ai     1.0000    0.8

{'test_score': 0.9340044742729307}

# ASSIGNMENT 2

In this assignment we will be using the trained model in the previous step to tag some texts.

**NOTE**: If you use the Basque corpus, you can get a document to process in Basque from a newspaper: https://www.berria.eus/

+ TODO: Pick a document of your choice and run the following Flair components:
  + Tokenize and segment the document into sentences.
  + Instiantiate a SequenceTagger with the generated lemmatizer model.
  + Lemmatize the sentences and print the results (ideally saving the predictions into a list of Sentence objects).

In [66]:
from flair.data import Sentence
from flair.models import SequenceTagger
import syntok.segmenter as segmenter

# 1. Load the model you just trained
tagger = SequenceTagger.load('/content/drive/My Drive/Colab Notebooks/2026-ILTAPP/en-lemma-model/best-model.pt')

# 2. Load your article
file_path = '/content/drive/My Drive/Colab Notebooks/2026-ILTAPP/2026-ILTAPP/Gaurdian_article.txt'
with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"{'Word':15} | {'Decoded Lemma'}")
print("-" * 35)

# 3. Process the text
for paragraph in segmenter.analyze(text):
    for sentence in paragraph:
        # Create Flair sentence
        flair_sentence = Sentence([token.value for token in sentence])

        # Predict SES rules
        tagger.predict(flair_sentence)

        # Decode rules into actual words
        for token in flair_sentence:
            predicted_rule = token.get_label('lemma').value

            try:
                # Apply the rule to the word to get the lemma
                lemma = _apply_lemma_rule(token.text, predicted_rule)
            except (ValueError, AssertionError):
                # Fallback if the rule is malformed
                lemma = token.text.lower()

            print(f"{token.text:15} | {lemma}")


2026-03-04 15:56:37,507 SequenceTagger predicts: Dictionary with 104 tags: <unk>, ↓0;d¦+, ↑0¦↓1;d¦+, ↓0;d¦-+, ↓0;abe, ↓0;d¦--+, ↓0;d¦---+, ↓0;d--+b¦+, ↑0¦↓-1;d¦+, ↓0;d¦-+v+e+, ↓0;d¦---+e+, ↓0;d¦-+o→+, ↓0;d¦--+e+, ↓0;d¦--+y+, ↑0¦↓1;ai, ↓0;d¦---+y+, ↓0;awe, ↓0;d¦-+y+, ↓0;d-+b¦--+, ↓0;d¦----+, ↓0;d¦--+o+, ↓0;d+s¦-+, ↓0;d-+w+i¦+, ↓0;d-+h+a¦+, ↓0;d¦-+e→+, ↓0;d¦--+a→+e+, ↓0;d¦-+k→+, ↓0;d→--+i¦+, ↓0;d¦--+a+v+e+, ↓0;ago, ↓0;d→-+i¦+, ↓0;d→-+o¦+, ↓0;d¦+n+, ↓0;d¦-----+i+n+k+, ↓0;d¦+e→-+, ↓0;d+'¦+, ↓0;d¦-+i+l+l+, ↓0;d¦-+d+, ↓0;d¦-+i→+, ↓0;d¦-+e→-+l+, ↑0¦↓1¦↑2¦↓3;d¦+, ↓0;d¦--+i+n→--+, ↑0¦↓1¦↑2¦↓-2;d¦+, ↓0;a", ↓0;d¦-+a→+, ↓0;d¦+o→+, ↓0;d¦-+u→+, ↓0;d-+'¦+, ↓0;a-, ↓0;d¦+a→+
Word            | Decoded Lemma
-----------------------------------
Iran            | Iran
war             | war
heralds         | herald
era             | era
of              | of
AI              | Ai
powered         | powered
bombing         | bombing
quicker         | quicker
than            | than
‘               | ‘
speed     

# ASSIGNMENT 3

As you can see in the previous step, the model we trained predicts this weird lemma_rules based on the minimum script edits generated by the get_ses_affixes.py script (in the resources folder). In order to obtain the real lemma we need to decode the lemma_rule using the function _apply_lemma_rule(form, lemma_rule) below.

+ This function takes as parameters the original word we want to lemmatize and the lemma_rule predicted by our lemmatizer model.

  ```
  decoded_lemma = _apply_lemma_rule('partidua', '↓0;d¦-+')
  print(decoded_lemma)
  'partidu'
  ```
Ideally you may have saved the previous predictions in a list of Sentence objects. Being that the case, you need to:

+ TODO: Iterate over the sentences to extract the lemma prediction (or lemma_rule) which will be used, together with the word, as input to obtain the decoded lemma.
+ TODO: print the results with the decoded lemma.


In [67]:
def _apply_lemma_rule(form, lemma_rule):
    if ';' not in lemma_rule:
        raise ValueError('lemma_rule %r for form %r missing semicolon' % (lemma_rule, form))
    casing, rule = lemma_rule.split(";", 1)
    if rule.startswith("a"):
        lemma = rule[1:]
    else:
        form = form.lower()
        rules, rule_sources = rule[1:].split("¦"), []
        assert len(rules) == 2
        for rule in rules:
            source, i = 0, 0
            while i < len(rule):
                if rule[i] == "→" or rule[i] == "-":
                    source += 1
                else:
                    assert rule[i] == "+"
                    i += 1
                i += 1
            rule_sources.append(source)

        try:
            lemma, form_offset = "", 0
            for i in range(2):
                j, offset = 0, (0 if i == 0 else len(form) - rule_sources[1])
                while j < len(rules[i]):
                    if rules[i][j] == "→":
                        lemma += form[offset]
                        offset += 1
                    elif rules[i][j] == "-":
                        offset += 1
                    else:
                        assert (rules[i][j] == "+")
                        lemma += rules[i][j + 1]
                        j += 1
                    j += 1
                    # print(lemma)
                if i == 0:
                    lemma += form[rule_sources[0]: len(form) - rule_sources[1]]
        except:
            lemma = lemma

    for rule in casing.split("¦"):
        if rule == "↓0": continue  # The lemma is lowercased initially
        if not rule: continue  # Empty lemma might generate empty casing rule
        case, offset = rule[0], int(rule[1:])
        lemma = lemma[:offset] + (lemma[offset:].upper() if case == "↑" else lemma[offset:].lower())
    return lemma


In [74]:
from flair.data import Sentence
from flair.models import SequenceTagger
import syntok.segmenter as segmenter

# Load the model you trained in Assignment 1
model_path = '/content/drive/My Drive/Colab Notebooks/2026-ILTAPP/en-lemma-model/best-model.pt'
tagger = SequenceTagger.load(model_path)


# Process the article text
print(f"{'Word':15} | {'Decoded Lemma'}")
print("-" * 35)

for paragraph in segmenter.analyze(text):
    for sentence in paragraph:
        # Create Flair sentence from tokens
        flair_sentence = Sentence([token.value for token in sentence])

        # ACTUALLY PREDICT THE TAGS
        tagger.predict(flair_sentence)

        for token in flair_sentence:
            word = token.text
            # Use the label name from your training ('lemma')
            lemma_rule = token.get_label('lemma').value

            try:
                # Apply the SES rule to get the base form
                decoded_lemma = _apply_lemma_rule(word, lemma_rule)
            except:
                decoded_lemma = word.lower() # Fallback

            print(f"{word:15} | {decoded_lemma}")


2026-03-04 16:02:39,598 SequenceTagger predicts: Dictionary with 104 tags: <unk>, ↓0;d¦+, ↑0¦↓1;d¦+, ↓0;d¦-+, ↓0;abe, ↓0;d¦--+, ↓0;d¦---+, ↓0;d--+b¦+, ↑0¦↓-1;d¦+, ↓0;d¦-+v+e+, ↓0;d¦---+e+, ↓0;d¦-+o→+, ↓0;d¦--+e+, ↓0;d¦--+y+, ↑0¦↓1;ai, ↓0;d¦---+y+, ↓0;awe, ↓0;d¦-+y+, ↓0;d-+b¦--+, ↓0;d¦----+, ↓0;d¦--+o+, ↓0;d+s¦-+, ↓0;d-+w+i¦+, ↓0;d-+h+a¦+, ↓0;d¦-+e→+, ↓0;d¦--+a→+e+, ↓0;d¦-+k→+, ↓0;d→--+i¦+, ↓0;d¦--+a+v+e+, ↓0;ago, ↓0;d→-+i¦+, ↓0;d→-+o¦+, ↓0;d¦+n+, ↓0;d¦-----+i+n+k+, ↓0;d¦+e→-+, ↓0;d+'¦+, ↓0;d¦-+i+l+l+, ↓0;d¦-+d+, ↓0;d¦-+i→+, ↓0;d¦-+e→-+l+, ↑0¦↓1¦↑2¦↓3;d¦+, ↓0;d¦--+i+n→--+, ↑0¦↓1¦↑2¦↓-2;d¦+, ↓0;a", ↓0;d¦-+a→+, ↓0;d¦+o→+, ↓0;d¦-+u→+, ↓0;d-+'¦+, ↓0;a-, ↓0;d¦+a→+
Word            | Decoded Lemma
-----------------------------------
Iran            | Iran
war             | war
heralds         | herald
era             | era
of              | of
AI              | Ai
powered         | powered
bombing         | bombing
quicker         | quicker
than            | than
‘               | ‘
speed     

# BONUS ASSIGNMENT 4

+ TODO Obtain manually the distance between two strings using the Levenshtein distance algorithm we presented in class for the following two words:

````
mellifluous -> mellow
````

You can write the matrix here using markdown.

## Bonus Assignment 4 — Levenshtein distance: `mellifluous` → `mellow`

|   |    | **m** | **e** | **l** | **l** | **o** | **w** |
|---|----|-------|-------|-------|-------|-------|-------|
|   | 0  | 1     | 2     | 3     | 4     | 5     | 6     |
| **m** | 1  | 0     | 1     | 2     | 3     | 4     | 5     |
| **e** | 2  | 1     | 0     | 1     | 2     | 3     | 4     |
| **l** | 3  | 2     | 1     | 0     | 1     | 2     | 3     |
| **l** | 4  | 3     | 2     | 1     | 0     | 1     | 2     |
| **i** | 5  | 4     | 3     | 2     | 1     | 1     | 2     |
| **f** | 6  | 5     | 4     | 3     | 2     | 2     | 2     |
| **l** | 7  | 6     | 5     | 4     | 3     | 3     | 3     |
| **u** | 8  | 7     | 6     | 5     | 4     | 4     | 4     |
| **o** | 9  | 8     | 7     | 6     | 5     | 4     | 5     |
| **u** | 10 | 9     | 8     | 7     | 6     | 5     | 5     |
| **s** | 11 | 10    | 9     | 8     | 7     | 6     | 6     |

**Levenshtein distance = 6**

The path involves matching `m-e-l-l` (4 matches), substituting `i→o`, then deleting `f`, `l`, `u`, `o`, `u`, `s` while inserting `w` — net 6 operations.